# 6.1 — BW Sales KPI Dashboard

## 1. Overview

This notebook builds a Tableau-ready extract from the `bw_sales_kpis` feed — a flat CSV that mimics what an SAP BW InfoCube delivers when it is exported for downstream BI tools (Tableau, Power BI, Looker).

**Input:** `data/raw/bw_sales_kpis.csv` (registered as the `bw_kpis` view).

**Output:** `data/tableau/bw_sales_kpis.hyper` — a Tableau extract ready for Tableau Desktop / Tableau Public.

**Workflow:** load → clean → analytical layer (DuckDB SQL) → visualize → export.

## 2. Load & inspect

In [ ]:
from analytics_bootcamp.duckdb_utils import get_connection, query_df
import pandas as pd

con = get_connection()
df = query_df("SELECT * FROM bw_kpis", con)
print(df.shape)
df.head()

In [ ]:
df.dtypes

## 3. Clean

Cast types, fill nulls in numeric columns, ensure month is treated as a string key.

In [ ]:
df = df.copy()
df["CAL_YEAR_MONTH"] = df["CAL_YEAR_MONTH"].astype(str)
for col in ["REVENUE", "QUANTITY", "RETURNS_QTY", "RETURNS_VALUE",
             "COGS", "GROSS_MARGIN", "DISCOUNT_AMT", "NUM_ORDERS"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
df.isna().sum()

## 4. Analytical layer

Build the metrics in DuckDB so the same SQL can be reused in the extract-generation script.

In [ ]:
monthly = con.sql("""
    SELECT
        CAL_YEAR_MONTH,
        SUM(REVENUE)      AS revenue,
        SUM(GROSS_MARGIN) AS margin
    FROM bw_kpis
    GROUP BY CAL_YEAR_MONTH
    ORDER BY CAL_YEAR_MONTH
""").df()
monthly.head()

In [ ]:
margin_by_org = con.sql("""
    SELECT
        VKORG,
        SUM(REVENUE)      AS revenue,
        SUM(GROSS_MARGIN) AS margin,
        ROUND(SUM(GROSS_MARGIN) / NULLIF(SUM(REVENUE), 0) * 100, 2) AS margin_pct
    FROM bw_kpis
    GROUP BY VKORG
    ORDER BY revenue DESC
""").df()
margin_by_org

In [ ]:
top_materials = con.sql("""
    SELECT
        MATNR,
        SUM(REVENUE) AS revenue
    FROM bw_kpis
    GROUP BY MATNR
    ORDER BY revenue DESC
    LIMIT 10
""").df()
top_materials

In [ ]:
# Month-over-month growth using LAG
mom = con.sql("""
    WITH m AS (
        SELECT CAL_YEAR_MONTH, SUM(REVENUE) AS revenue
        FROM bw_kpis
        GROUP BY CAL_YEAR_MONTH
    )
    SELECT
        CAL_YEAR_MONTH,
        revenue,
        LAG(revenue) OVER (ORDER BY CAL_YEAR_MONTH) AS prev_revenue,
        ROUND(
            (revenue - LAG(revenue) OVER (ORDER BY CAL_YEAR_MONTH))
            / NULLIF(LAG(revenue) OVER (ORDER BY CAL_YEAR_MONTH), 0) * 100,
            2
        ) AS mom_growth_pct
    FROM m
    ORDER BY CAL_YEAR_MONTH
""").df()
mom

## 5. Visualize

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(monthly["CAL_YEAR_MONTH"], monthly["revenue"], marker="o")
ax.set_title("Monthly Revenue Trend")
ax.set_xlabel("Calendar Month")
ax.set_ylabel("Revenue")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(margin_by_org["VKORG"].astype(str), margin_by_org["margin_pct"])
ax.set_title("Gross Margin % by Sales Org")
ax.set_xlabel("Sales Org (VKORG)")
ax.set_ylabel("Margin %")
plt.tight_layout()
plt.show()

## 6. Export to .hyper

In [ ]:
from analytics_bootcamp.tableau import to_hyper

extract = con.sql("""
    SELECT
        CAL_YEAR_MONTH,
        VKORG,
        MATNR,
        SUM(REVENUE)      AS total_sales,
        SUM(GROSS_MARGIN) AS total_margin,
        ROUND(SUM(GROSS_MARGIN) / NULLIF(SUM(REVENUE), 0) * 100, 2) AS margin_pct
    FROM bw_kpis
    GROUP BY CAL_YEAR_MONTH, VKORG, MATNR
""").df()

to_hyper(extract, "data/tableau/bw_sales_kpis.hyper", "BW_Sales_KPIs")

## 7. Tableau tips

Once the `.hyper` file is connected in Tableau Desktop / Tableau Public:

**Suggested dimensions**
- `CAL_YEAR_MONTH` — drag to Columns; convert to a continuous date if needed.
- `VKORG` (Sales Org), `MATNR` (Material) — for filters and drill-downs.

**Suggested measures**
- `total_sales` (SUM)
- `total_margin` (SUM)
- `margin_pct` (AVG, or compute on the fly: SUM(margin)/SUM(sales))

**Recommended chart types**
- Revenue trend: line chart with `CAL_YEAR_MONTH` on Columns and `total_sales` on Rows.
- Margin by Sales Org: horizontal bar with `VKORG` and `margin_pct`.
- Top materials: bar chart of `MATNR` filtered to the top 10 by revenue.

**Dashboard layout idea**
- Top row: KPI tiles (Total revenue, Total margin, Margin %).
- Middle row: revenue trend with a sales-org filter.
- Bottom row: top-10 material bar chart with action filters into the trend chart.